Step 1: Load and prepare documents

## Document loaders in LangChain

Document loaders ingest content from various sources and convert it into LangChain `Document` objects. Each `Document` typically includes `page_content` (the text) and optional `metadata` (source, page number, URL, etc.).

### Common loader types
- **WebBaseLoader**: Load and parse web pages from URLs.
- **TextLoader**: Load plain text files.
- **PDFLoader** variants: Extract text from PDFs.
- **CSVLoader**: Load CSV rows into documents.
- **DirectoryLoader**: Batch-load files from a folder using a file pattern.

### Typical workflow
1. Initialize a loader with a source (file path, URL, or directory).
2. Call `.load()` to get a list of `Document` objects.
3. Optionally split documents into chunks before indexing or retrieval.

### Example (conceptual)
- Initialize a loader with a URL or file
- Call `.load()` to retrieve documents
- Pass the documents into a text splitter and vector store

Use document loaders to standardize data ingestion across different sources before downstream tasks like chunking, embedding, and retrieval.

In [ ]:
%run langchain_common_notebook.py

In [ ]:
enable_logging()

In [ ]:
# List of URLs to load documents from
urls = ["https://lilianweng.github.io/posts/2023-06-23-agent/",
        "https://lilianweng.github.io/posts/2023-06-25-agent-reasoning/"]

# Load documents from the URLs
docs = [WebBaseLoader(
                url, 
                requests_kwargs={"headers": {"User-Agent": os.getenv("USER_AGENT")}}
            ).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

Step 2: Split documents into chunks

In [ ]:
# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

In [ ]:
print(f"Total chunks in doc_splits: {len(doc_splits)}\n")

# Show a quick preview of the first 3 chunks
for i, doc in enumerate(doc_splits[:3], start=1):
    print(f"--- Chunk {i} ---")
    print("Metadata:", doc.metadata)
    print("Content preview:", doc.page_content[:200], "...")
    print()

Step 3: Create a vector store

In [ ]:
vectorstore = SKLearnVectorStore.from_documents(
    documents=doc_splits,
    embedding=databricks_embeddings,
)
      
retriever = vectorstore.as_retriever(k=4)
print("\n✅ Vector store created successfully!")

Step 4: Set up the LLM and prompt template

In [ ]:
# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"],
)

# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

In [ ]:
disable_logging()

### Step 5: Invoke the chain

In [ ]:
import pprintpp

question_a = "What are the key components of an AI agent?"
question_b = "what is the weather in lahore"

question = question_a

# Retrieve relevant documents
documents = retriever.invoke(question)

# Extract content from retrieved documents
doc_texts = "\n".join([doc.page_content for doc in documents])
print(doc_texts[:1000])  # Show a preview of the retrieved documents


In [ ]:
# Get the answer from the language model
answer = rag_chain.invoke({"question": question, "documents": doc_texts})

print(f"\nQuestion: {question}\n")
print(f"Answer: {answer}")

Step 5: Integrate the retriever and LLM into a RAG application

In [ ]:
# Define a class to encapsulate the retriever and RAG chain

class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
        
    def run(self, question):
        # Retrieve relevant documents
        documents = self.retriever.invoke(question)
        
        # Extract content from retrieved documents
        doc_texts = "\\n".join([doc.page_content for doc in documents])
        
        # Get the answer from the language model
        answer = self.rag_chain.invoke({"question": question, "documents": doc_texts})
        return answer


Step 6: Test your RAG application

In [ ]:
# Create and test the RAG application
if 'vectorstore' in locals() and vectorstore is not None and 'rag_chain' in locals():
    rag_app = RAGApplication(retriever, rag_chain)

    # Test with some questions
    questions = [
        "What are the key components of an AI agent?",
        "How do adversarial attacks work on large language models?"
    ]

    # Test each question
    for question in questions:
        print(f"\n🤖 Question: {question}")
        print("-" * 50)
        try:
            answer = rag_app.run(question)
            print(f"📋 Answer: {answer}")
        except Exception as e:
            print(f"❌ Error: {e}")
        print("=" * 50)
else:
    print("🚨 Cannot test RAG application - setup incomplete.")
    print("Please run the previous cells to fix configuration issues.")

### `RunnablePassthrough` and `RunnableLambda` (quick intro)

- **`RunnablePassthrough`**: Forwards input unchanged.  
    Useful when a value (like the user question) should move through a chain while other fields are computed in parallel.

- **`RunnableLambda`**: Wraps a Python function as a runnable step.  
    Useful for lightweight transformations, such as formatting retrieved documents before prompting.

```python
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

chain = {
        "question": RunnablePassthrough(),              # keep original input
        "context": retriever | RunnableLambda(format_docs),  # transform docs -> text
}
```

In short: **Passthrough keeps data as-is; Lambda applies custom logic.**

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Better RAG chain: retrieval + formatting + generation in one LCEL pipeline
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

better_prompt = PromptTemplate(
    template="""You are a precise QA assistant.
Use only the context below to answer the question.
If the answer is not in the context, say "I don't know."
Keep the response to at most 3 sentences.

Question: {question}
Context:
{context}

Answer:""",
    input_variables=["question", "context"],
)

# In this LCEL pipeline, the input is the user question (a plain string).
# - "question": RunnablePassthrough() forwards that original question unchanged.
# - "context": retriever gets relevant docs for the same question, then format_docs
#   converts those documents into one context string for the prompt.
# The prompt then receives both fields: {question, context}.
better_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
    }
    | better_prompt
    | llm
    | StrOutputParser()
)

print(better_rag_chain.invoke("What are the key components of an AI agent?"))